# Feature Selection Experiment Results

## Objective

Evaluate the impact of different feature selection techniques on multiple machine learning algorithms while comparing different resampling strategies. The objective is to identify the most effective preprocessing pipeline before hyperparameter tuning.

The evaluated feature selection methods include:

- SelectFdr (multiple significance levels)
- SelectFromModel (Gradient Boosting)
- SelectFromModel (Logistic Regression)

Models were evaluated using **Average Precision (AP)** on cross-validation, training, and validation datasets.

## Selected Models

Rather than tuning every model, representative models from different algorithm families were selected based on their performance characteristics.

| Model | Pipeline | Selection Reason |
|-------|----------|------------------|
| **Gradient Boosting** | RandomOverSampler + SelectFdr (α = 0.005) | Highest validation Average Precision (0.090930). |
| **Logistic Regression (Lasso)** | Baseline + SelectFdr (α = 0.05) | Best-performing linear model with minimal overfitting. |
| **Random Forest** | Baseline + SelectFromModel (Gradient Boost, Threshold = 1.5 × Mean) | Extremely high training AP (0.735592), indicating severe overfitting. Selected to investigate whether hyperparameter tuning can improve generalization. |
| **XGBoost** | RandomOverSampler + SelectFromModel (Logistic Regression, Threshold = 1.5 × Mean) | Best-performing XGBoost configuration despite moderate performance. Included because XGBoost is widely recognized as a strong baseline for structured (tabular) datasets and may benefit substantially from hyperparameter optimization. |

## Summary of Selected Pipelines

| Model | CV AP | Train AP | Validation AP |
|------|------:|---------:|--------------:|
| Gradient Boosting | 0.102609 | 0.117770 | **0.090930** |
| Logistic Regression (Lasso) | 0.105889 | 0.086149 | **0.086107** |
| Random Forest | 0.058055 | **0.735592** | 0.051690 |
| XGBoost | 0.080374 | 0.102420 | 0.082011 |

## Observations

- **Gradient Boosting** achieved the highest validation Average Precision among all evaluated models. However, the noticeable gap between training and validation performance indicates moderate overfitting.

- **Logistic Regression** consistently produced the smallest gap between training and validation Average Precision, demonstrating excellent generalization despite having slightly lower predictive performance.

- **Random Forest** exhibited severe overfitting, with a training Average Precision exceeding 0.73 while achieving only 0.05 on the validation set. Although its current predictive performance is poor, Random Forest remains a strong ensemble algorithm whose complexity can often be controlled through hyperparameter tuning.

- **XGBoost** did not outperform Gradient Boosting in the current experiments. Nevertheless, XGBoost is widely regarded as one of the strongest algorithms for tabular data and frequently achieves significant improvements after careful hyperparameter optimization. Therefore, it is retained as a candidate for tuning.

## Conclusion

Four representative models were selected for hyperparameter optimization:

- Gradient Boosting
- Logistic Regression (Lasso)
- Random Forest
- XGBoost

Each model represents a different learning paradigm and exhibits distinct strengths and weaknesses:

- Gradient Boosting provides the highest predictive performance.
- Logistic Regression offers the best generalization.
- Random Forest demonstrates high learning capacity but suffers from severe overfitting.
- XGBoost serves as a competitive gradient boosting baseline commonly used for structured datasets.

The next stage focuses on hyperparameter tuning to improve validation performance while reducing overfitting, particularly for the tree-based ensemble models.

In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
from pathlib import Path
import sys

project_root = Path.cwd().parents[1]
sys.path.insert(0, str(project_root))

In [3]:
READ_CSV="../../data/interim/data_travel_insurance_interim.csv"
SAVE_RESULT = "results/feature_selections_optimization.csv"
RANDOM_STATE=42


TARGET_TRANSFORM_COLS = ["Destination"]
LOG_TRANSFORM_COLS= ["Duration", "Net Sales"]

In [4]:
from src.utils import benchmark_models

import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, OneHotEncoder, TargetEncoder
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import PowerTransformer
from sklearn.feature_selection import SelectFdr, SelectFromModel, f_classif

from xgboost import XGBClassifier

from feature_engine.outliers import Winsorizer

from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.pipeline import Pipeline as ImbPipeline

In [5]:
df = pd.read_csv(READ_CSV)
df.head()

,Agency,Agency Type,Distribution Channel,Product Name,Duration,Destination,Net Sales,Commission,Age,Claim,Is Refund,Suspected Fraud,Commission Rate
0,C2B,Airlines,Online,Annual Silver Plan,365,SINGAPORE,216.0,54.0,57,0,No,No,0.25
1,EPX,Travel Agency,Online,Cancellation Plan,4,MALAYSIA,10.0,0.0,33,0,No,No,0.00
2,JZI,Airlines,Online,Basic Plan,19,INDIA,22.0,7.7,26,0,No,No,0.35
3,EPX,Travel Agency,Online,2 way Comprehensive Plan,20,UNITED STATES,112.0,0.0,59,0,No,No,0.00
4,C2B,Airlines,Online,Bronze Plan,8,SINGAPORE,16.0,4.0,28,0,No,No,0.25


In [6]:
x = df.drop(columns=["Claim"])
y = df["Claim"]


In [7]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

In [8]:
NUMERIC_COLS = [features for features in x_train.columns if x_train[features].dtypes != "O"]
CATEGORICAL_COLS = [features for features in x_train.columns if x_train[features].dtypes == "O"]


In [9]:
numeric_pipeline = Pipeline([
    ("winsorizer_iqr", Winsorizer(capping_method="iqr", fold=1.5)),
    ("RobustScaler", RobustScaler()),
])

numeric_log_pipeline = Pipeline([
    ("power", PowerTransformer(method="yeo-johnson")),
    ("RobustScaler", RobustScaler()),
])

categorical_ohe_pipeline = Pipeline([
     ("OneHotEncoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False, drop="first"))
])

categorical_target_pipeline = Pipeline([
     ("TargetEncoder", TargetEncoder())
])

In [10]:
preprocessor = ColumnTransformer(
    [
       
        ("numeric_pipeline", numeric_pipeline, [c for c in NUMERIC_COLS if c not in LOG_TRANSFORM_COLS]),
        ("numeric_log_pipeline", numeric_log_pipeline, LOG_TRANSFORM_COLS),
        ("categorical_ohe_pipeline", categorical_ohe_pipeline, [c for c in CATEGORICAL_COLS if c not in TARGET_TRANSFORM_COLS]),
        ("categorical_target_pipeline", categorical_target_pipeline, TARGET_TRANSFORM_COLS),

    ],
    remainder="drop"
)

In [11]:
pipeline = ImbPipeline([
    ("preprocessor", preprocessor),
    ("feature_selection", "passthrough"),
    ("resampler", "passthrough"),
    ("classifier", LogisticRegression())
])

In [12]:
logreg_base = LogisticRegression(random_state=RANDOM_STATE)
logreg_lasso = LogisticRegression(penalty="l1", solver="liblinear", random_state=RANDOM_STATE)
logreg_ridge = LogisticRegression(penalty="l2", solver="saga", random_state=RANDOM_STATE)
logreg_elasticnet = LogisticRegression(penalty="elasticnet", solver="saga", l1_ratio=0.5, random_state=RANDOM_STATE)
rf = RandomForestClassifier(random_state=RANDOM_STATE)
gb = GradientBoostingClassifier(random_state=RANDOM_STATE)
xgb = XGBClassifier(random_state=RANDOM_STATE)


In [13]:
feature_selectors = {
    "None": "passthrough",
    "FDR_0.002": SelectFdr(score_func=f_classif, alpha=0.002),
    "FDR_0.003": SelectFdr(score_func=f_classif, alpha=0.003),
    "FDR_0.004": SelectFdr(score_func=f_classif, alpha=0.004),
    "FDR_0.005": SelectFdr(score_func=f_classif, alpha=0.005),
    "FDR_0.01": SelectFdr(score_func=f_classif, alpha=0.01),
    "FDR_0.05": SelectFdr(score_func=f_classif, alpha=0.05),
    "FDR_0.25": SelectFdr(score_func=f_classif, alpha=0.25),
    "SFM_GB_1.5mean": SelectFromModel(
        GradientBoostingClassifier(random_state=RANDOM_STATE),
        threshold="1.5*mean"
    )
}

In [14]:
models = {
    "LogisticRegressionBase": logreg_base,
    "LogisticRegressionLasso": logreg_lasso,
    "LogisticRegressionRidge": logreg_ridge,
    "LogisticRegressionElasticNet": logreg_elasticnet,
    "RandomForest": rf,
    "GradientBoost": gb,
    "XGBoost": xgb
}

In [15]:
ros = RandomOverSampler(random_state=RANDOM_STATE)

strategies = {
    "Baseline": ("passthrough", models),
    "RandomOverSampler": (ros, models),
}

In [16]:
results = pd.DataFrame()

for fs_name, selector in feature_selectors.items():

    pipeline.set_params(feature_selection=selector)

    for strategy_name, (resampler, model_dict) in strategies.items():

        pipeline.set_params(resampler=resampler)

        result = benchmark_models(
            pipeline=pipeline,
            list_model=model_dict,
            x_train=x_train,
            y_train=y_train,
            x_test=x_test,
            y_test=y_test,
            cv=5,
            postfix=f"{strategy_name}_{fs_name}"
        )

        results = pd.concat([results, result], ignore_index=True)

In [17]:
results = results.sort_values("mean_ap_validate_score", ascending=False)
results[0:10]

,name,ap_test_score,mean_ap_train_score,mean_ap_validate_score,f1_test_score,recall_test_score,precision_test_score,accuracy_test_score
77,GradientBoost_RandomOverSampler_FDR_0.01,0.101167,0.120699,0.089668,0.116859,0.711111,0.063660,0.815347
63,GradientBoost_RandomOverSampler_FDR_0.005,0.103204,0.116762,0.089209,0.123116,0.725926,0.067261,0.822347
35,GradientBoost_RandomOverSampler_FDR_0.003,0.100902,0.112438,0.088538,0.118582,0.718519,0.064624,0.816493
28,GradientBoost_Baseline_FDR_0.003,0.103083,0.206474,0.088255,0.000000,0.000000,0.000000,0.982566
21,GradientBoost_RandomOverSampler_FDR_0.002,0.108090,0.115346,0.087613,0.119363,0.666667,0.065550,0.831000
49,GradientBoost_RandomOverSampler_FDR_0.004,0.099405,0.114951,0.087544,0.120073,0.733333,0.065390,0.815347
91,GradientBoost_RandomOverSampler_FDR_0.05,0.102262,0.120822,0.086676,0.116618,0.740741,0.063291,0.807203
84,LogisticRegressionLasso_Baseline_FDR_0.05,0.105889,0.086148,0.086105,0.000000,0.000000,0.000000,0.982820
0,LogisticRegressionElasticNet_Baseline_None,0.107269,0.086275,0.086035,0.000000,0.000000,0.000000,0.982820
1,LogisticRegressionLasso_Baseline_None,0.105234,0.086822,0.085809,0.000000,0.000000,0.000000,0.982820


In [19]:
results.to_csv(SAVE_RESULT, index=False)